# [실습 12-3] RAG 미니 체험 — 문서고 열람 권한 (저장소 전용)

| 항목 | 내용 |
|---|---|
| 예상 소요 시간 | 20~30분 (CPU 충분) |
| 본문 연계 | 12.4.2 RAG — 지면에는 이 노트북 안내(QR)만 있고 코드는 없다 |
| 선수 실습 | [실습 12-1] ([보조 2] 임베딩 유사도가 준비 운동) |

"우리 회사 규정을 물어보니 그럴듯한 거짓말을 한다"(장 도입 훅) —
**검색 → 증강 → 생성** 3단계를 벡터 DB 없이 넘파이 코사인
유사도만으로 최소 구현해, 같은 질문에 대한 RAG 유/무 답변을
나란히 비교한다.

### [준비] 설치와 모델 로드 (저장소 전용)

In [ ]:
# !pip -q install transformers sentence-transformers
import platform
import numpy as np
from sentence_transformers import SentenceTransformer

emb = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2")
print("Python", platform.python_version())
print("임베딩 모델 준비 완료 (다국어 경량)")

# 생성 모델: 실습 12-2의 Qwen 재사용
import torch
from transformers import (AutoTokenizer,
                          AutoModelForCausalLM)
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype="auto", device_map="auto")

def ask(prompt, max_new=192, seed=42):
    torch.manual_seed(seed)
    text = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**ids, max_new_tokens=max_new,
                       do_sample=True, temperature=0.3)
    return tok.decode(
        out[0][ids["input_ids"].shape[1]:],
        skip_special_tokens=True).strip()

### [보조 1] 문서고 만들기 — 가상 사내 규정

In [ ]:
DOCS = {
 "휴가 규정 §3": ("정규직의 연차 휴가는 입사 첫해 "
   "11일, 2년차 15일이며, 3년차부터 "
   "3년 단위로 1일씩 가산된다"
   "(3~5년차 16일, 6~8년차 17일). "
   "최대 25일이다."),
 "휴가 규정 §7": ("연차는 반차(0.5일) 단위로 나눠 "
   "쓸 수 있고, 사용 기한은 발생 연도 말일까지다."),
 "복리후생 §2": ("점심 식대는 1일 12,000원 한도로 "
   "법인카드 사용을 원칙으로 한다."),
 "복리후생 §5": ("자기계발비는 연 100만 원 한도로 "
   "도서·강의·자격증 응시료에 사용할 수 있다."),
 "근무 규정 §4": ("재택근무는 주 2일까지 팀장 승인으로 "
   "가능하며, 코어타임은 10시부터 16시까지다."),
 "근무 규정 §9": ("초과 근무는 사전 신청제이며, "
   "야간(22시 이후) 근무는 원칙적으로 금지한다."),
}
print(f"문서 {len(DOCS)}편 — 우리 회사만의 지식이라 "
      "LLM은 이 내용을 모른다")

### [보조 2] 검색 — 질문과 가까운 문서 찾기

In [ ]:
names = list(DOCS)
doc_vecs = emb.encode([DOCS[n] for n in names])

def retrieve(question, k=3):
    """질문 벡터와 코사인 유사도 상위 k편 반환."""
    q = emb.encode([question])[0]
    sims = doc_vecs @ q / (
        np.linalg.norm(doc_vecs, axis=1)
        * np.linalg.norm(q))
    top = np.argsort(sims)[::-1][:k]
    for i in top:
        print(f"  유사도 {sims[i]:.3f}  {names[i]}")
    return [names[i] for i in top]

Q = "입사 4년차인데 연차가 며칠인가요?"
print(f"질문: {Q}")
picked = retrieve(Q)

**왜 상위 2편이 아니라 3편(k=3)인가**  
의미 검색(임베딩)은 정답 문단을 **1위로 올려 주지 않는다**. 실제로 이 질문에서는 정작 답이 담긴 「휴가 규정 §3」이 **1위가 아니라 2위**이고, 연차 *사용법*만 담은 「휴가 규정 §7」이 1위, 심지어 **무관한 「근무 규정 §4」(재택근무)까지 상위 3편에 섞여** 든다 — 질의어(‘4년차·연차’)와 각 문단의 표층 어휘가 어긋나기 때문이다. 그래서 정답 문단을 놓치지 않도록(**recall 확보**) 검색 편수 `k`를 여유 있게 잡는다. 실무에서도 `k=3~5`가 기본값이다 — 검색이 완벽하다고 가정하지 않는 것이 RAG 설계의 출발점이다.

**핵심 포인트**: 키워드가 아니라 **뜻의 거리**로 찾는다 — "연차가 며칠"이라는 질문이 '휴가 규정'을 골라내는 것은 임베딩 공간에서 둘이 가깝기 때문이다(12.1.3 → 12.4.2). 이것이 RAG의 "문서고 사서" 단계다.

### [보조 3] 증강·생성 — RAG 없음 vs RAG 있음

In [ ]:
print("═══ RAG 없음 — 맨몸의 LLM ═══")
print(ask(Q), "\n")

context = "\n".join(f"[{n}] {DOCS[n]}"
                     for n in picked)
P_RAG = (f"아래 사내 규정 문서에서만 근거를 찾아 "
         f"답하라. 문서에 없으면 '문서에 없음'이라고 "
         f"답하라.(12.3 프롬프트 원칙의 응용)\n\n"
         f"{context}\n\n질문: {Q}")
print("═══ RAG 있음 — 문서고 열람 후 ═══")
print(ask(P_RAG))

**핵심 포인트**
- RAG 없음: 그럴듯한 일반론 또는 지어낸 숫자(**할루시네이션의 실물**, 12.4.1) — 모델은 우리 회사 규정을 배운 적이 없다.
- RAG 있음: 검색된 조항을 근거로 "첫해 11일, 2년차 15일, 3년마다 +1일 → 4년차 16일" 계산까지 시도한다 — 문서고 열람 권한을 준 효과.
- "문서에서만 답하라"는 지시 자체가 12.3 프롬프트 원칙의 응용이다 — 이 지시가 없으면 모델이 문서를 무시하고 아는 척을 시작한다.

실패 시 대처: 답변이 문서를 무시하면 위 지시문이 프롬프트에 있는지 확인한다.

### [심화 1] 문서 추가·질문 변형 (연습문제 심화 프로젝트형 직결)

In [ ]:
# TODO 1: DOCS에 새 규정(예: 경조사 휴가)을 추가하고
#         관련 질문으로 검색이 잘 찾는지 확인하자.
# TODO 2: 문서에 없는 질문("주차 지원되나요?")을 던져
#         '문서에 없음' 답변이 나오는지 검증하자 —
#         모른다고 말하게 만드는 것이 RAG 설계의 핵심.
# TODO 3: k=1로 줄이면 어떤 질문에서 실패하는지 찾고,
#         검색 품질과 답변 품질의 관계를 논증하자.
Q2 = "주차비 지원이 되나요?"
print(f"질문: {Q2}")
picked2 = retrieve(Q2)
print(ask((f"아래 문서에서만 근거를 찾아 답하라. "
           f"없으면 '문서에 없음'이라고 답하라.\n\n"
           + "\n".join(f"[{n}] {DOCS[n]}"
                        for n in picked2)
           + f"\n\n질문: {Q2}")))

---
## 마무리

- RAG = 검색(뜻이 가까운 문서) + 증강(프롬프트에 첨부) + 생성(근거 기반 답변) — 3단계가 각각 코드 몇 줄이었다.
- 프레임워크·벡터 DB는 이 원리의 **규모 확장판**일 뿐이다 — 원리를 알면 도구가 바뀌어도 두렵지 않다.
- "모르면 모른다고 답하게 하라" — 할루시네이션 대책의 절반은 프롬프트 설계다(12.3 ↔ 12.4의 연결).

**연습문제 연계**: [심화 프로젝트형] "사내 문서 QA 시스템 RAG 설계" 문항의 출발 뼈대가 이 노트북 전체다 — [심화 1]의 TODO 3종을 확장하면 된다.